In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

forecast = pd.read_csv(
    r'C:\Users\Ashut\Retailpulse\data\processed\demand_forecast.csv'
)
forecast['ds'] = pd.to_datetime(forecast['ds'])

future_forecast = forecast[
    forecast['ds'] > forecast['ds'].max() - pd.Timedelta(days=30)
][['ds','yhat','yhat_lower','yhat_upper']].copy()

future_forecast.columns = ['date','forecast',
                            'forecast_lower',
                            'forecast_upper']
future_forecast[['forecast','forecast_lower',
                 'forecast_upper']] = \
    future_forecast[['forecast','forecast_lower',
                     'forecast_upper']].clip(lower=0)

print(f"✓ Forecast loaded: {future_forecast.shape}")
print(future_forecast.head())

SAFETY_FACTOR = 1.1
future_forecast['reorder_qty'] = (
    future_forecast['forecast_upper'] * SAFETY_FACTOR
).round(0).astype(int)

LOW_STOCK_THRESHOLD = future_forecast['forecast'].mean() * 0.3
future_forecast['alert'] = future_forecast.apply(
    lambda row: '🔴 LOW STOCK'
    if row['forecast_lower'] < LOW_STOCK_THRESHOLD
    else '🟢 OK', axis=1
)

print("\n=== Inventory Recommendations ===")
print(future_forecast[['date','forecast',
    'reorder_qty','alert']].to_string(index=False))

future_forecast.to_csv(
    r'C:\Users\Ashut\Retailpulse\data\processed\inventory_recommendations.csv',
    index=False
)

✓ Forecast loaded: (30, 4)
          date      forecast  forecast_lower  forecast_upper
709 2011-11-10  50084.335453    36019.839793    63284.160175
710 2011-11-11  38074.353943    24319.079092    51725.769066
711 2011-11-12  12657.590024        0.000000    27536.756505
712 2011-11-13  31334.731544    18448.018012    45376.575938
713 2011-11-14  41385.260011    27349.909125    54798.342268

=== Inventory Recommendations ===
      date     forecast  reorder_qty       alert
2011-11-10 50084.335453        69613        🟢 OK
2011-11-11 38074.353943        56898        🟢 OK
2011-11-12 12657.590024        30290 🔴 LOW STOCK
2011-11-13 31334.731544        49914        🟢 OK
2011-11-14 41385.260011        60278        🟢 OK
2011-11-15 47871.722053        68013        🟢 OK
2011-11-16 45473.761978        65437        🟢 OK
2011-11-17 54311.420450        75283        🟢 OK
2011-11-18 43009.677819        62318        🟢 OK
2011-11-19 18273.811461        36735 🔴 LOW STOCK
2011-11-20 37663.761820        57